# Этап 6 V4 — сравнение TabM с базовой моделью XGBoost

## Исследовательский вопрос

**Сохраняется ли предварительно слабый результат TabM из V3 на полном 3-fold OOF и способна ли TabM как самостоятельная модель превзойти сохранённую базовую модель XGBoost при том же наборе из 47 разрешённых признаков?**

### Почему проверяем это сейчас?

В V3 TabM была проверена только на одном внешнем фолде и показала предварительно более слабый результат, чем XGBoost. Одного фолда недостаточно, чтобы считать этот результат устойчивым.

Поэтому в V4 выполняем полный 3-fold OOF-прогон и проверяем, сохраняется ли наблюдавшаяся разница на всей рабочей выборке.

### Что меняется?

Только объём проверки: вместо одного внешнего фолда выполняются все три фолда и формируется полный OOF-прогноз.

### Что остаётся неизменным?

- датасет `Data_final.xlsb`;
- target `DefMark`;
- идентификатор `INN`;
- те же 47 разрешённых признаков;
- `Q_B1_norm` и `Q_B2_norm` не используются как predictors;
- разбиение данных и seed;
- конфигурация TabM;
- preprocessing;
- метрики;
- отсутствие tuning, balancing и threshold optimization;
- final test не используется.

Для сравнения заранее фиксируем шесть OOF-метрик: `Gini`, `ROC-AUC`, `PR-AUC`, `Precision`, `Recall` и `F1`.

Следующая code-ячейка только подготавливает единый вывод итогового сравнения TabM и XGBoost. Она не выполняет обучение и не меняет экспериментальный протокол.

In [11]:
def show_v4_oof_comparison():
    if globals().get('om') is None or globals().get('od') is None: return
    keys=('Gini','ROC-AUC','PR-AUC','Precision','Recall','F1')
    rows_html=''.join(f'<tr><td>{key}</td><td>{float(XGB_OOF[key]):.5f}</td><td>{om[key]:.5f}</td><td>{od[key]:+.5f}</td></tr>' for key in keys)
    display(HTML(f'<h3>Полный OOF: TabM и XGBoost</h3><table><tr><th>Метрика</th><th>XGBoost</th><th>TabM</th><th>Δ TabM − XGBoost</th></tr>{rows_html}</table>'))


## Почему нужен полный OOF-прогон?

Предыдущая версия V3 завершилась после одного внешнего фолда и дала предварительно неудовлетворительный результат TabM относительно XGBoost.

Этот результат важен как сигнал, но одного фолда недостаточно для окончательного вывода.

В V4 мы не пытаемся улучшать TabM и не подбираем новые параметры. Задача этого эксперимента намного уже: **проверить, сохраняется ли результат V3 на полном 3-fold OOF при полностью неизменном ML-протоколе.**

Время выполнения измеряется как характеристика модели, но не используется как ограничение эксперимента.

## 1. Фиксируем неизменный контракт эксперимента

### Что проверяем?

Перед обучением фиксируем датасет, рабочую выборку, порядок 47 разрешённых признаков, конфигурацию TabM и пути сохранения результатов V4.

### Зачем?

Сравнение с V3 и базовой моделью XGBoost имеет смысл только в том случае, если данные и экспериментальный протокол остаются неизменными.

### Как это отвечает на исследовательский вопрос?

Код проверяет контрольную сумму датасета, рабочее разбиение и точный набор разрешённых признаков до начала обучения. Это защищает эксперимент от случайного изменения данных или feature set.

### Что остаётся неизменным?

`Data_final.xlsb`, `DefMark`, `INN`, 47 разрешённых признаков, исключение `Q_B1_norm` и `Q_B2_norm`, разбиение данных, seed, конфигурация TabM и отсутствие подбора гиперпараметров.

In [12]:
from __future__ import annotations
import hashlib,json,os,platform,random,time,traceback
from pathlib import Path
import numpy as np,pandas as pd,tabm,torch,torch.nn.functional as F
from IPython.display import HTML,display
from sklearn.metrics import average_precision_score,f1_score,precision_score,recall_score,roc_auc_score
from sklearn.model_selection import StratifiedKFold,train_test_split
from torch.utils.data import DataLoader,TensorDataset
def root():
    for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
        if (p/'pyproject.toml').exists(): return p
    raise FileNotFoundError('Не найден корень проекта.')
def file_hash(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def index_hash(x): return hashlib.sha256(np.asarray(x,dtype=np.int64).tobytes()).hexdigest()
def seed(s): random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.use_deterministic_algorithms(True)
ROOT=root();DATASET=ROOT/'data'/'raw'/'Data_final.xlsb';BASELINE_PATH=ROOT/'reports'/'generated'/'stage1_baseline_results_V2.json'
GENERATED_DIR=ROOT/'reports'/'generated';SUMMARY_DIR=ROOT/'reports'/'summary';RESULTS_PATH=GENERATED_DIR/'stage6_tabm_results_V4.json';SUMMARY_PATH=SUMMARY_DIR/'stage6_tabm_summary_V4.json';OOF_PATH=GENERATED_DIR/'stage6_tabm_oof_V4.npz'
DATA_HASH='fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930';WORK_HASH='80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45';TARGET='DefMark';IDENTIFIER='INN';FORBIDDEN=['Q_B1_norm','Q_B2_norm'];SEED=42;FOLD_SEEDS={1:43,2:44,3:45};MAX_EPOCHS=100;PATIENCE=16
CFG={'arch_type':'tabm','k':8,'n_blocks':3,'d_block':512,'activation':'ReLU','dropout':0.1,'start_scaling_init':'random-signs','d_out':2,'lr':0.002,'weight_decay':0.0003,'betas':(0.9,0.999),'eps':1e-8,'gradient_clip_global_norm':1.0,'batch_size':256}
CFG.update({'num_embeddings':None,'input_dtype':'float32','optimizer':'AdamW','share_training_batches':True,'max_epochs':MAX_EPOCHS,'amp':False,'torch_compile':False,'scheduler':None,'warmup':None,'class_weights':None,'sampling':None})
baseline=json.loads(BASELINE_PATH.read_text(encoding='utf-8'));FEATURES=baseline['допустимые_признаки'];XGB=baseline['модели']['XGBoost'];XGB_OOF=XGB['итоговые_метрики'];XGB_FOLDS=XGB['метрики_фолдов']
if len(FEATURES)!=47 or any(x in FEATURES for x in FORBIDDEN): raise ValueError('Нарушен контракт 47 разрешённых признаков.')


## 2. Готовим обучение и понятное отображение прогресса

### Что проверяем?

Определяем модель TabM, процедуру выбора числа эпох, повторное обучение на полном train-фолде, расчёт метрик и единую панель прогресса.

### Зачем?

Полный CPU-прогон занимает много времени, поэтому выполнение должно быть наблюдаемым и не выглядеть зависшим.

### Как это отвечает на исследовательский вопрос?

Для каждого внешнего фолда число эпох определяется только на внутренней validation-части, после чего модель заново обучается на полном train-фолде и строит прогноз для внешнего фолда. В результате мы получаем сопоставимые OOF-предсказания для всех трёх фолдов.

### Что остаётся неизменным?

Архитектура TabM, optimizer, learning rate, batch size, early stopping и остальные параметры обучения сохраняются такими же, как в зафиксированном протоколе V3.

In [13]:
CURRENT=0;DISPLAY=None
def since(t): return time.perf_counter()-t
def show(stage,e,total,fs,all0,best=None,best_e=None,avg=None,eta=None):
    global DISPLAY
    et='—' if e is None else f'{e}/{total}';pct='—' if e is None else f'{round(100*e/total)}%';auc='пока нет' if best is None or not np.isfinite(best) else f'{best:.5f}';av='появится после первой эпохи' if avg is None else f'{avg/60:.2f} мин';left='пока нельзя оценить' if eta is None else f'{max(0,eta)/60:.1f} мин'
    h=HTML(f'<div style="font-family:Arial;border:1px solid #bbb;padding:10px"><h3>Этап 6 V4</h3><b>Фолд:</b> {CURRENT}/3<br><b>Этап:</b> {stage}<br><b>Эпоха:</b> {et}; <b>Прогресс:</b> {pct}<br><b>Лучшая эпоха:</b> {best_e if best_e else "пока нет"}; <b>Лучший ROC-AUC:</b> {auc}<br><b>Время фолда:</b> {since(fs)/60:.1f} мин; <b>Общее время:</b> {since(all0)/60:.1f} мин<br><b>Среднее время эпохи:</b> {av}; <b>Осталось:</b> {left}</div>')
    if DISPLAY is None: DISPLAY=display(h,display_id=True)
    else: DISPLAY.update(h)
def net(n): return tabm.TabM.make(n_num_features=n,cat_cardinalities=None,arch_type=CFG['arch_type'],k=CFG['k'],n_blocks=CFG['n_blocks'],d_block=CFG['d_block'],activation=CFG['activation'],dropout=CFG['dropout'],start_scaling_init=CFG['start_scaling_init'],num_embeddings=None,d_out=CFG['d_out'])
def make_loader(x,y,s,shuffle): return DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(y.astype(np.int64))),batch_size=CFG['batch_size'],shuffle=shuffle,generator=torch.Generator(device='cpu').manual_seed(s),num_workers=0)
def train_epoch(m,l,o):
    m.train()
    for x,y in l:
        o.zero_grad(set_to_none=True);z=m(x.float());target=y[:,None].expand(-1,m.k).reshape(-1);F.cross_entropy(z.reshape(-1,2),target).backward();torch.nn.utils.clip_grad_norm_(m.parameters(),CFG['gradient_clip_global_norm']);o.step()
@torch.inference_mode()
def predict(m,x):
    m.eval();t=torch.from_numpy(x).float();return np.concatenate([torch.softmax(m(t[i:i+CFG['batch_size']]),dim=-1).mean(dim=1)[:,1].cpu().numpy() for i in range(0,len(t),CFG['batch_size'])])
def score(y,p):
    q=(p>=.5).astype(np.int64);a=float(roc_auc_score(y,p));return {'ROC-AUC':a,'Gini':2*a-1,'PR-AUC':float(average_precision_score(y,p)),'Precision':float(precision_score(y,q,zero_division=0)),'Recall':float(recall_score(y,q,zero_division=0)),'F1':float(f1_score(y,q,zero_division=0))}
def select(x,y,s,fs,all0):
    a,b=train_test_split(np.arange(len(y)),test_size=.1,stratify=y,random_state=s);seed(s);m=net(x.shape[1]);o=torch.optim.AdamW(m.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'],betas=CFG['betas'],eps=CFG['eps']);l=make_loader(x[a],y[a],s,True);be,ba,stale,times=0,float('-inf'),0,[]
    for e in range(1,MAX_EPOCHS+1):
        t=time.perf_counter();train_epoch(m,l,o);v=float(roc_auc_score(y[b],predict(m,x[b])));be,ba,stale=(e,v,0) if v>ba else (be,ba,stale+1);times.append(since(t));avg=float(np.mean(times));show('выбор числа эпох',e,MAX_EPOCHS,fs,all0,ba,be,avg,avg*(MAX_EPOCHS-e))
        if stale>=PATIENCE: break
    return be,ba,e
def refit(x,y,v,s,epochs,ba,fs,all0):
    seed(s);m=net(x.shape[1]);o=torch.optim.AdamW(m.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'],betas=CFG['betas'],eps=CFG['eps']);l=make_loader(x,y,s,True);times=[]
    for e in range(1,epochs+1):
        t=time.perf_counter();train_epoch(m,l,o);times.append(since(t));avg=float(np.mean(times));show('полное обучение фолда',e,epochs,fs,all0,ba,epochs,avg,avg*(epochs-e))
    show('прогноз',epochs,epochs,fs,all0,ba,epochs);return predict(m,v),e


## 3. Выполняем полный контролируемый эксперимент

### Что проверяем?

Последовательно выполняем все три внешних фолда, собираем единый OOF-прогноз TabM и сравниваем его с сохранённым OOF-результатом XGBoost.

### Зачем?

Нужно проверить, был ли слабый результат V3 особенностью одного фолда или он сохраняется на всей рабочей выборке.

### Как это отвечает на исследовательский вопрос?

После завершения трёх фолдов получаем:

- метрики каждого отдельного фолда;
- полный OOF-результат TabM;
- разницу между TabM и XGBoost по заранее зафиксированным метрикам;
- фактическое время выполнения.

### Что остаётся неизменным?

Используется тот же `StratifiedKFold`, seed `42`, seeds фолдов `43/44/45`, тот же набор признаков и тот же ML-протокол.

Final test, tuning, class weights, balancing и threshold optimization в этом эксперименте не используются.

## 4. Защищаем длинный запуск от потери прогресса

Полный прогон TabM занимает несколько часов, поэтому технически предусмотрены контрольные точки (`checkpoint`) и продолжение после прерывания (`resume`).

После каждой полностью завершённой эпохи сохраняются состояние модели, optimizer, генераторы случайных чисел, уже рассчитанные OOF-результаты и текущий прогресс.

Если выполнение прерывается, повторный запуск продолжает эксперимент с первой незавершённой операции. Максимальная потеря — только текущая незавершённая эпоха.

Важно: этот механизм **не меняет ML-протокол и не влияет на результат эксперимента**. Он нужен только для безопасного выполнения длительного расчёта.

После успешного формирования и проверки финальных артефактов технический checkpoint удаляется.

In [14]:
CHECKPOINT_PATH=ROOT/'reports'/'generated'/'stage6_tabm_checkpoint_V4.pt'
CHECKPOINT_VERSION='Stage 6 TabM V4 checkpoint v1'

def _feature_sha(): return hashlib.sha256('\n'.join(FEATURES).encode()).hexdigest()
def _contract(ds,wi):
    return {'experiment':'Stage 6','version':'V4','checkpoint_version':CHECKPOINT_VERSION,'dataset_sha256':ds,'working_index_sha256':wi,'feature_names_in_order':list(FEATURES),'feature_identity_sha256':_feature_sha(),'tabm_config':CFG,'outer_cv':{'type':'StratifiedKFold','n_splits':3,'shuffle':True,'random_state':SEED},'fold_seeds':FOLD_SEEDS,'early_stopping':{'inner_train_fraction':.9,'inner_validation_fraction':.1,'patience':PATIENCE,'selection_metric':'ROC-AUC'}}
def _atomic_torch(x,p):
    q=p.with_name(p.name+'.tmp');torch.save(x,q);os.replace(q,p)
def _atomic_json(x,p):
    q=p.with_name(p.name+'.tmp');q.write_text(json.dumps(x,ensure_ascii=False,indent=2),encoding='utf-8');os.replace(q,p)
def _atomic_npz(p,**x):
    q=p.with_name(p.stem+'.tmp.npz');np.savez_compressed(q,**x);os.replace(q,p)
def _rng(): return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state()}
def _load_checkpoint(p):
    try:return torch.load(p,map_location='cpu',weights_only=False)
    except TypeError:return torch.load(p,map_location='cpu')
def _restore_rng(x): random.setstate(x['python']);np.random.set_state(x['numpy']);torch.set_rng_state(x['torch'])
def _loader(x,y,s,gs=None):
    g=torch.Generator(device='cpu');g.manual_seed(s)
    if gs is not None:g.set_state(gs)
    return DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(y.astype(np.int64))),batch_size=CFG['batch_size'],shuffle=True,generator=g,num_workers=0),g
def _phase(p): return {'selection':'выбор числа эпох','refit':'полное обучение','inference':'прогноз и метрики','between_folds':'между фолдами'}[p]
def _total(): return RUNTIME_BEFORE_SESSION+time.perf_counter()-SESSION_STARTED
def _epoch(s): return s['selection_epoch'] if s['phase']=='selection' else s['refit_epoch']
def _progress(s,status='running',error=None):
    return {'experiment':'Stage 6','version':'V4','status':status,'current_fold':s.get('current_fold'),'current_phase':s.get('phase'),'completed_epochs':{'selection':s.get('selection_epoch',0),'refit':s.get('refit_epoch',0),'last_completed_epoch':s.get('last_completed_epoch',0)},'completed_fold_metrics':s.get('rows',[]),'oof_assignments_completed':int(np.count_nonzero(s.get('oof_f',[]))),'runtime_seconds':_total(),'last_checkpoint_unix':s.get('last_checkpoint_unix'),'error':error}
def _panel(s,fs,saved=False):
    global DISPLAY,CURRENT
    CURRENT=s['current_fold'];e=_epoch(s);n=MAX_EPOCHS if s['phase']=='selection' else s.get('refit_epochs',0)
    durations=s['selection_epoch_seconds'] if s['phase']=='selection' else s['refit_epoch_seconds']
    avg=float(np.mean(durations)) if durations else None;pct='—' if not n else f'{round(100*e/n)}%';eta='пока нельзя оценить' if avg is None or not n else f'{max(0,avg*(n-e))/60:.1f} мин';avg_text='появится после первой эпохи' if avg is None else f'{avg/60:.2f} мин'
    auc=s.get('best_inner_roc_auc',-np.inf);auc='пока нет' if not np.isfinite(auc) else f'{auc:.5f}';last='ещё не создан' if not s.get('last_checkpoint_unix') else f'{time.time()-s["last_checkpoint_unix"]:.0f} с назад'
    h=HTML(f'<div style="font-family:Arial;border:1px solid #bbb;padding:10px"><h3>Этап 6 V4 — checkpoint/resume</h3><b>Режим:</b> {s["mode"]}<br><b>Фолд:</b> {CURRENT}/3<br><b>Этап:</b> {_phase(s["phase"])}<br><b>Эпоха:</b> {e}/{n or "—"}; <b>Прогресс:</b> {pct}<br><b>Лучшая эпоха:</b> {s.get("best_epoch") or "пока нет"}; <b>Лучший ROC-AUC:</b> {auc}<br><b>Среднее время эпохи:</b> {avg_text}; <b>Осталось:</b> {eta}<br><b>Время фолда:</b> {(time.perf_counter()-fs)/60:.1f} мин; <b>Время текущей сессии:</b> {(time.perf_counter()-SESSION_STARTED)/60:.1f} мин; <b>Общее время:</b> {_total()/60:.1f} мин<br><b>Последний checkpoint:</b> {last}{"<br><b>✓ Эпоха сохранена</b>" if saved else ""}</div>')
    if DISPLAY is None:DISPLAY=display(h,display_id=True)
    else:DISPLAY.update(h)
def _save(s,fs,m=None,o=None,g=None):
    s['runtime_seconds']=_total();s['fold_runtime_seconds']=time.perf_counter()-fs;s['last_checkpoint_unix']=time.time();s['last_completed_epoch']=_epoch(s);s['model_state_dict']=None if m is None else m.state_dict();s['optimizer_state_dict']=None if o is None else o.state_dict();s['dataloader_generator_state']=None if g is None else g.get_state();s['rng_state']=_rng()
    _atomic_torch(s,CHECKPOINT_PATH);_atomic_json(_progress(s),RESULTS_PATH);_panel(s,fs,True)
def _training(x,y,s,st):
    # Fresh phase повторяет исходный V4 path: seed -> model -> optimizer -> training.
    resume_phase=st.get('model_state_dict') is not None and st.get('optimizer_state_dict') is not None and _epoch(st)>0
    seed(s);m=net(x.shape[1]);o=torch.optim.AdamW(m.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'],betas=CFG['betas'],eps=CFG['eps'])
    if resume_phase:m.load_state_dict(st['model_state_dict']);o.load_state_dict(st['optimizer_state_dict'])
    l,g=_loader(x,y,s,st.get('dataloader_generator_state') if resume_phase else None)
    if resume_phase:_restore_rng(st['rng_state'])
    return m,o,l,g
def _validate(st,c):
    keys=('experiment','version','checkpoint_version','dataset_sha256','working_index_sha256','feature_names_in_order','feature_identity_sha256','tabm_config','outer_cv','fold_seeds','early_stopping');bad=[k for k in keys if st.get('contract',{}).get(k)!=c.get(k)]
    if bad:raise ValueError('Checkpoint несовместим с текущим контрактом эксперимента: '+', '.join(bad)+'. Продолжение остановлено.')
def _new(c,n):
    return {'contract':c,'mode':'новый запуск','current_fold':1,'phase':'selection','selection_epoch':0,'refit_epoch':0,'refit_epochs':0,'last_completed_epoch':0,'best_epoch':0,'best_inner_roc_auc':-np.inf,'patience_counter':0,'model_state_dict':None,'optimizer_state_dict':None,'dataloader_generator_state':None,'rng_state':_rng(),'rows':[],'oof_p':np.full(n,np.nan,np.float32),'oof_f':np.zeros(n,np.int8),'selection_epoch_seconds':[],'refit_epoch_seconds':[],'runtime_seconds':0.,'fold_runtime_seconds':0.,'last_checkpoint_unix':None}
def _selection(x,y,s,st,fs):
    a,b=train_test_split(np.arange(len(y)),test_size=.1,stratify=y,random_state=s)
    if st['selection_epoch']<MAX_EPOCHS and st['patience_counter']<PATIENCE:
        m,o,l,g=_training(x[a],y[a],s,st)
        for e in range(st['selection_epoch']+1,MAX_EPOCHS+1):
            started=time.perf_counter();train_epoch(m,l,o);v=float(roc_auc_score(y[b],predict(m,x[b])))
            if v>st['best_inner_roc_auc']:st['best_epoch']=e;st['best_inner_roc_auc']=v;st['patience_counter']=0
            else:st['patience_counter']+=1
            st['selection_epoch']=e;st['selection_epoch_seconds'].append(time.perf_counter()-started);_save(st,fs,m,o,g)
            if st['patience_counter']>=PATIENCE:break
    st.update({'phase':'refit','refit_epoch':0,'refit_epochs':st['best_epoch'],'last_completed_epoch':0,'model_state_dict':None,'optimizer_state_dict':None,'dataloader_generator_state':None});_save(st,fs)
def _refit(x,y,s,st,fs):
    m,o,l,g=_training(x,y,s,st)
    for e in range(st['refit_epoch']+1,st['refit_epochs']+1):
        started=time.perf_counter();train_epoch(m,l,o);st['refit_epoch']=e;st['refit_epoch_seconds'].append(time.perf_counter()-started);_save(st,fs,m,o,g)
    st['phase']='inference';_save(st,fs,m,o,g)
def _final(st,y):
    om=score(y,st['oof_p']);base={k:float(XGB_OOF[k]) for k in ('ROC-AUC','Gini','PR-AUC','Precision','Recall','F1')};od={k:om[k]-base[k] for k in base};decision='Облегчённая TabM не дала прироста к сильному GBDT baseline.' if od['Gini']<=-.01 and od['PR-AUC']<=-.005 else 'Результат требует review по полным OOF-метрикам и времени.'
    meta={**st['contract'],'status':'completed','target':TARGET,'identifier':IDENTIFIER,'final_test_used':False,'fold_metrics':st['rows'],'oof_metrics':om,'oof_complete':True,'runtime_seconds':_total(),'comparison_vs_xgboost':{'oof_delta':od,'fold_deltas':[r['delta_vs_xgboost'] for r in st['rows']]},'versions':{'python':platform.python_version(),'torch':torch.__version__,'tabm':tabm.__version__,'numpy':np.__version__,'cpu_count':os.cpu_count()},'limitations':['Random CV не доказывает temporal stability.','3 folds не являются statistical significance claim.','Порог 0.5 — диагностический.','Final test не использован.'],'early_stopping':{'inner_train_fraction':.90,'inner_validation_fraction':.10,'patience':PATIENCE,'selection_metric':'ROC-AUC','refit_epochs':'best_epoch'},'decision':decision,'error':None}
    summary={'experiment':'Stage 6','version':'V4','status':'completed','oof_metrics':om,'delta_vs_xgboost':od,'runtime_seconds':_total(),'decision':decision}
    return meta,summary,om,od
def run_v4_checkpointed():
    global SESSION_STARTED,RUNTIME_BEFORE_SESSION,DISPLAY,om,od
    SESSION_STARTED=time.perf_counter();RUNTIME_BEFORE_SESSION=0.;DISPLAY=None;om=od=None;st=None
    try:
        GENERATED_DIR.mkdir(parents=True,exist_ok=True);SUMMARY_DIR.mkdir(parents=True,exist_ok=True);ds=file_hash(DATASET)
        if ds!=DATA_HASH:raise ValueError('Контрольная сумма данных не совпадает.')
        data=pd.read_excel(DATASET,engine='pyxlsb');allowed=[z for z in data.columns if z not in [TARGET,IDENTIFIER,*FORBIDDEN]]
        if allowed!=FEATURES:raise ValueError('Порядок разрешённых признаков не совпадает.')
        x=data.loc[:,FEATURES].to_numpy(np.float32);y=data[TARGET].to_numpy(np.int64)
        if not np.isfinite(x).all():raise ValueError('Обнаружены пропуски или нечисловые значения.')
        idx=np.arange(len(data));work,hold=train_test_split(idx,test_size=.2,stratify=y,random_state=SEED);wi=index_hash(work)
        if len(work)!=289614 or wi!=WORK_HASH:raise ValueError('Рабочее разбиение не совпадает.')
        xw,yw=x[work],y[work];del data,x,y,idx,hold;c=_contract(ds,wi)
        if CHECKPOINT_PATH.exists():
            st=_load_checkpoint(CHECKPOINT_PATH);_validate(st,c);st.setdefault('selection_epoch_seconds',[]);st.setdefault('refit_epoch_seconds',[]);st['mode']='resume (восстановление checkpoint)';RUNTIME_BEFORE_SESSION=float(st['runtime_seconds']);print(f'Найден сохранённый Stage 6 V4. Фолд: {st["current_fold"]} из 3; этап: {_phase(st["phase"])}; завершено эпох: {_epoch(st)}; накопленное время: {RUNTIME_BEFORE_SESSION/60:.1f} мин. Продолжаем со следующей операции.')
        else:st=_new(c,len(yw));st['mode']='fresh run (новый запуск)';fresh_fold_started=time.perf_counter();print('Stage 6 V4: новый запуск с fold 1.');_save(st,fresh_fold_started)
        splits=list(StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED).split(xw,yw));first=st['current_fold']+(st['phase']=='between_folds')
        for fold in range(first,4):
            tr,va=splits[fold-1];s=FOLD_SEEDS[fold]
            if fold!=st['current_fold'] or st['phase']=='between_folds':st.update({'current_fold':fold,'phase':'selection','selection_epoch':0,'refit_epoch':0,'refit_epochs':0,'last_completed_epoch':0,'best_epoch':0,'best_inner_roc_auc':-np.inf,'patience_counter':0,'model_state_dict':None,'optimizer_state_dict':None,'dataloader_generator_state':None,'selection_epoch_seconds':[],'refit_epoch_seconds':[],'fold_runtime_seconds':0.});new_fold_started=time.perf_counter();_save(st,new_fold_started)
            fs=time.perf_counter()-st['fold_runtime_seconds']
            if st['phase']=='selection':_selection(xw[tr],yw[tr],s,st,fs)
            if st['phase']=='refit':_refit(xw[tr],yw[tr],s,st,fs)
            if st['phase']=='inference':
                m,_,_,_=_training(xw[tr],yw[tr],s,st);p=predict(m,xw[va]);fm=score(yw[va],p);st['oof_p'][va]=p;st['oof_f'][va]=fold;base=XGB_FOLDS[fold-1];d={k:fm[k]-float(base[k]) for k in fm};st['rows'].append({'fold':fold,'seed':s,'best_epoch':st['best_epoch'],'inner_best_roc_auc':st['best_inner_roc_auc'],'selection_epochs':st['selection_epoch'],'refit_epochs':st['refit_epoch'],'runtime_seconds':time.perf_counter()-fs,'metrics':fm,'delta_vs_xgboost':d});st.update({'phase':'between_folds','model_state_dict':None,'optimizer_state_dict':None,'dataloader_generator_state':None});_save(st,fs)
        meta,summary,om,od=_final(st,yw);_atomic_json(meta,RESULTS_PATH);_atomic_json(summary,SUMMARY_PATH);_atomic_npz(OOF_PATH,y_true=yw,oof_probability=st['oof_p'],fold=st['oof_f'])
        if not (RESULTS_PATH.exists() and SUMMARY_PATH.exists() and OOF_PATH.exists()):raise RuntimeError('Не удалось проверить запись финальных V4 artifacts.')
        CHECKPOINT_PATH.unlink();print(f'Этап 6 V4 завершён. Суммарное время: {_total()/60:.1f} мин.')
    except KeyboardInterrupt:
        error={'тип':'KeyboardInterrupt','сообщение':'Запуск прерван пользователем.'}
        if st is not None:_atomic_json(_progress(st,'interrupted',error),RESULTS_PATH)
        print('Запуск прерван. Последний checkpoint сохранён.')
    except Exception as exc:
        error={'тип':type(exc).__name__,'сообщение':str(exc),'трассировка':traceback.format_exc()}
        if st is not None:_atomic_json(_progress(st,'error',error),RESULTS_PATH)
        raise


In [15]:
run_v4_checkpointed()
show_v4_oof_comparison()


Stage 6 V4: новый запуск с fold 1.


Этап 6 V4 завершён. Суммарное время: 522.2 мин.


Метрика,XGBoost,TabM,Δ TabM − XGBoost
Gini,0.80399,0.78131,-0.02268
ROC-AUC,0.90199,0.89066,-0.01134
PR-AUC,0.59927,0.56799,-0.03128
Precision,0.71226,0.73760,+0.02535
Recall,0.36845,0.28085,-0.08760
F1,0.48566,0.40680,-0.07886


# Результат исследования

Полный 3-fold OOF-прогон завершён. Теперь можно ответить на исследовательский вопрос по всей рабочей выборке, а не по результату одного фолда.

## ФАКТЫ

Эксперимент успешно завершён:

- выполнены все 3 из 3 внешних фолдов;
- суммарное время выполнения — `31 332.65 сек` (≈ `8 ч 42 мин`);
- финальная тестовая выборка не использовалась.

### Результаты по отдельным фолдам

| Фолд | Gini | ROC-AUC | PR-AUC | Precision | Recall | F1 |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 0.775686 | 0.887843 | 0.564407 | 0.748570 | 0.266224 | 0.392764 |
| 2 | 0.784749 | 0.892375 | 0.563632 | 0.731917 | 0.273078 | 0.397754 |
| 3 | 0.789785 | 0.894892 | 0.577825 | 0.733299 | 0.303244 | 0.429058 |

### Итоговый OOF-результат TabM

- `Gini = 0.781310`
- `ROC-AUC = 0.890655`
- `PR-AUC = 0.567993`
- `Precision = 0.737602`
- `Recall = 0.280850`
- `F1 = 0.406804`

### Разница относительно базовой модели XGBoost

- `Gini = -0.022680`
- `ROC-AUC = -0.011340`
- `PR-AUC = -0.031280`
- `Precision = +0.025347`
- `Recall = -0.087596`
- `F1 = -0.078857`


## ИНТЕРПРЕТАЦИЯ

Предварительный результат предыдущей версии эксперимента подтвердился на полном 3-fold OOF.

**TabM как самостоятельная модель не превзошла сохранённую базовую модель XGBoost при текущем протоколе с 47 разрешёнными признаками.**

TabM показала более низкие значения `Gini`, `ROC-AUC`, `PR-AUC`, `Recall` и `F1`.

Единственная из сравниваемых метрик, по которой TabM показала более высокий результат, — `Precision`.

Таким образом, полученные результаты не дают оснований заменять сильную бустинговую модель XGBoost на самостоятельную TabM.

Особенно заметна разница по `Recall`: TabM обнаруживает меньшую долю объектов дефолтного класса. Для нашей задачи это существенный недостаток, поскольку способность находить дефолты является одной из важных характеристик модели.

При этом результат относится именно к TabM как к **самостоятельной модели** в зафиксированном экспериментальном протоколе. Он не означает, что TabM не может оказаться полезной при другом способе её использования.

## ОГРАНИЧЕНИЯ

- Случайная кросс-валидация и OOF-оценка не доказывают временную устойчивость модели.
- Результаты трёх фолдов сами по себе не доказывают статистическую значимость различий между моделями.
- Порог классификации `0.5` используется только как диагностический и отдельно не оптимизировался.
- Финальная тестовая выборка в исследовании не использовалась.
- Подбор гиперпараметров TabM в рамках этого эксперимента не проводился.
- Эксперимент проверяет TabM только как самостоятельную модель.
- Эксперимент не отвечает на вопрос, способна ли TabM извлечь дополнительную полезную информацию из прогнозов сильных бустинговых моделей.

## СЛЕДУЮЩИЙ ШАГ

Мы получили ответ на текущий исследовательский вопрос: **самостоятельная TabM в зафиксированном 47-признаковом протоколе уступает базовой модели XGBoost.**

Поэтому следующий исследовательский вопрос меняет не данные и не базовый протокол, а способ использования TabM:

**Может ли TabM показать более полезный результат, если использовать её не вместо бустинговых моделей, а поверх их прогнозов?**

Следующий notebook должен проверить это как отдельный контролируемый эксперимент: использовать выходы бустинговых моделей как входную информацию для TabM и определить, появляется ли за счёт такой комбинации дополнительная прогностическая ценность.

Текущий эксперимент этого вопроса не проверяет.